# IEEE-CIS Fraud Detection: 비용 민감형 사기탐지 모델 개발
## Day 6. 하이퍼파라미터 튜닝 (Optuna)

**목표**
- RF, LightGBM, XGBoost 3개 모델을 Optuna로 튜닝하여 기본 설정 대비 성능 향상 확인
- 튜닝 후 최종 모델 선정 및 CalibratedClassifierCV 확률 보정 적용

**튜닝 전략**
- 도구: Optuna (베이지안 최적화 기반)
- CV: StratifiedGroupKFold 5-fold (Day 4와 동일, 공정한 비교를 위해 동일 조건 유지)
- 평가 지표: PR-AUC (핵심 지표, 불균형 데이터에서 더 의미 있음)
- Trial 수: RF 30 / LightGBM 50 / XGBoost 50
- 조기 종료: 적용 (불필요한 trial 낭비 방지)

### 참고: Optuna 하이퍼파라미터 튜닝 개념 정리

---

#### Optuna란

하이퍼파라미터 최적화를 자동으로 수행하는 파이썬 라이브러리. Grid Search(모든 조합 탐색)나 Random Search(무작위 탐색)와 달리, 베이지안 최적화 기반으로 이전 trial 결과를 반영하여 다음 탐색 방향을 지능적으로 결정함.

**핵심 구성 요소**
- **Study**: 하나의 최적화 실험 단위 (예: "LightGBM 튜닝")
- **Trial**: Study 안의 개별 시도. 하이퍼파라미터를 하나 뽑아서 학습 후 성능을 평가하는 1회 과정
- **Objective 함수**: 하이퍼파라미터 조합을 입력받아 성능(PR-AUC)을 반환하는 함수. Optuna가 이 값을 최대화하는 방향으로 trial을 설계함

---

#### Grid Search / Random Search와의 비교

| 방식 | 탐색 전략 | 장점 | 단점 |
|---|---|---|---|
| Grid Search | 모든 조합 전부 시도 | 완전 탐색 | 조합이 많으면 시간이 기하급수적으로 증가 |
| Random Search | 무작위로 조합 시도 | Grid보다 빠름 | 이전 결과를 반영하지 않아 비효율적 |
| Optuna | 이전 결과 기반 베이지안 최적화 | 적은 trial로 효율적 탐색 | 구현이 다소 복잡 |

---

#### 조기 종료(Pruning)

각 trial 도중, 중간 성능이 기존 best trial보다 현저히 낮으면 해당 trial을 중단하고 다음 trial로 넘어가는 기능. 불필요한 trial에 시간을 낭비하지 않아 전체 튜닝 시간을 줄여줌.

```python
# Optuna Pruner 예시
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner()  # 중간값보다 낮으면 조기 종료
)
```

---

#### Trial 수 설정 근거

| 모델 | Trial 수 | 이유 |
|---|---|---|
| Random Forest | 30 | 탐색할 핵심 파라미터가 상대적으로 단순하고, 학습 속도가 느려 수익이 빨리 줄어듦 |
| LightGBM | 50 | learning_rate, num_leaves 등 탐색축이 많아 trial을 넉넉히 두는 것이 유리 |
| XGBoost | 50 | LightGBM과 동일한 이유, scale_pos_weight 조정도 포함 |

---

#### 면접 한 줄 설명
"Grid Search는 조합이 기하급수적으로 늘어나고, Random Search는 이전 결과를 반영하지 못해서, 이전 trial 결과를 베이지안 최적화로 반영하여 효율적으로 탐색하는 Optuna를 사용했습니다."

---

#### 튜닝 중 지표 운용 전략

**튜닝 단계(Optuna trial 중)**
- PR-AUC만 계산하여 최적화 (속도 확보)
- trial마다 ROC-AUC, F1 등을 같이 계산하면 trial당 시간이 늘어나 전체 튜닝 시간이 크게 증가함

**최종 best 파라미터 확정 후**
- PR-AUC(메인), ROC-AUC(보조)를 함께 확인
- Calibration 보정(CalibratedClassifierCV) 적용
- 4.9절 Threshold Optimization에서 비용함수 기반 최적 임계값 탐색 후 precision, recall, F1, confusion matrix 확인

**이유**
PR-AUC는 threshold-independent 지표로, 모델이 사기/정상을 얼마나 잘 순위 매기는지는 알려주지만 실제 운영 임계값까지는 바로 알려주지 않음. 따라서 튜닝은 PR-AUC로 하되, 최종 운영에서는 비즈니스 목표(FP/FN 비용)에 맞는 임계값을 별도로 설정하여 precision/recall을 함께 확인하는 것이 안전함.

**한 줄 정리**
불균형 데이터에서 PR-AUC를 튜닝 메인 지표로 쓰는 것은 적절하며 권장됨. 단 최종 운영에서는 임계값을 별도로 설정하여 precision/recall을 함께 확인함.

### 1. 라이브러리 임포트 및 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
import optuna
import pickle
import os
import sys
sys.path.append("..")

from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import average_precision_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)  # trial별 로그 최소화
pd.set_option('display.max_columns', 100)

# Day 3 최종 데이터
df = pd.read_parquet("../data/processed/day3_final.parquet")

# Day 4에서 확정한 변수 분류 재정의
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'UID', 'UID_v2']
target = 'isFraud'
groups = df['UID']

categorical_cols = df.drop(columns=exclude_cols).select_dtypes(include='category').columns.tolist()
numeric_cols = df.drop(columns=exclude_cols).select_dtypes(include=[np.number]).columns.tolist()
binary_cols = [c for c in numeric_cols if df[c].nunique() <= 2]
scale_cols = [c for c in numeric_cols if c not in binary_cols]

X = df.drop(columns=exclude_cols + [target])
y = df[target]

imbalance_ratio = (y == 0).sum() / (y == 1).sum()
print(f"shape: {df.shape}")
print(f"클래스 불균형 비율: 1:{imbalance_ratio:.1f}")
print(f"수치형(표준화 대상): {len(scale_cols)}개")
print(f"범주형: {len(categorical_cols)}개")

c:\Users\seonu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shape: (590540, 208)
클래스 불균형 비율: 1:27.6
수치형(표준화 대상): 149개
범주형: 49개


### 2. CV 함수 재정의 (튜닝용 경량 버전)

튜닝 중에는 PR-AUC만 계산하여 속도를 확보합니다. 
타겟 인코딩과 StandardScaler는 Day 4와 동일하게 fold 내부에서만 적용합니다.

In [3]:
from sklearn.preprocessing import StandardScaler

def target_encode(train_df, valid_df, cols, target_col, smoothing=20):
    global_mean = train_df[target_col].mean()
    encoded_train = train_df.copy()
    encoded_valid = valid_df.copy()
    for col in cols:
        encoded_train[col] = encoded_train[col].astype(str)
        encoded_valid[col] = encoded_valid[col].astype(str)
        stats = train_df.assign(**{col: train_df[col].astype(str)}).groupby(col)[target_col].agg(['mean', 'count'])
        smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        encoded_train[col] = encoded_train[col].map(smoothed).fillna(global_mean)
        encoded_valid[col] = encoded_valid[col].map(smoothed).fillna(global_mean)
    return encoded_train, encoded_valid


def cv_pr_auc(model, use_category=False):
    """
    튜닝용 경량 CV: PR-AUC만 반환.
    """
    sgkf = StratifiedGroupKFold(n_splits=5)
    pr_aucs = []

    for train_idx, val_idx in sgkf.split(X, y, groups=groups):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val   = y.iloc[val_idx]

        if not use_category:
            temp_train = pd.concat([X_train[categorical_cols], y_train], axis=1)
            temp_val   = X_val[categorical_cols].copy()
            temp_train_enc, temp_val_enc = target_encode(
                temp_train, temp_val, categorical_cols, target
            )
            X_train[categorical_cols] = temp_train_enc[categorical_cols].values
            X_val[categorical_cols]   = temp_val_enc[categorical_cols].values

        scaler = StandardScaler()
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_val[scale_cols]   = scaler.transform(X_val[scale_cols])

        if use_category:
            for col in categorical_cols:
                X_train[col] = X_train[col].astype('category')
                X_val[col]   = X_val[col].astype('category')
        else:
            for col in categorical_cols:
                X_train[col] = X_train[col].astype(float)
                X_val[col]   = X_val[col].astype(float)

        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]
        pr_aucs.append(average_precision_score(y_val, y_pred))

    return np.mean(pr_aucs)

### 3. Random Forest 튜닝 (15 trials, 경량화 버전)

**초기 설정 대비 변경 사항**
- n_trials: 30 → 15 (LightGBM 튜닝 결과(0.5101)가 RF 기본값(0.4972)을 이미 넘었으므로,
  "RF를 튜닝하면 LightGBM을 넘길 수 있는지 빠르게 확인"하는 용도로 축소)
- n_estimators: 50~200 고정 범위 → 300 고정 (early stopping 없이 적당한 고정값 사용,
  RF는 XGBoost/LightGBM과 달리 early stopping API가 표준화되어 있지 않음)
- max_depth: 5~30 → 5~15로 축소
- n_jobs=-1로 병렬 학습 최대화

**탐색 파라미터**
- max_depth: 트리 최대 깊이
- min_samples_split: 분기에 필요한 최소 샘플 수
- min_samples_leaf: 리프 노드 최소 샘플 수
- max_features: 각 분기에서 고려할 feature 비율

**고정 파라미터**
- n_estimators=300 (속도와 성능의 균형)
- class_weight='balanced'

In [ ]:
# def objective_rf(trial):
#     params = {
#         'n_estimators':      300,
#         'max_depth':         trial.suggest_int('max_depth', 5, 15),
#         'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
#         'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
#         'max_features':      trial.suggest_float('max_features', 0.3, 1.0),
#         'class_weight':      'balanced',
#         'n_jobs':            -1,
#         'random_state':      42,
#     }
#     model = RandomForestClassifier(**params)
#     return cv_pr_auc(model, use_category=False)


# study_rf = optuna.create_study(
#     direction='maximize',
#     pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1)
# )
# study_rf.optimize(objective_rf, n_trials=15, show_progress_bar=True)

# print(f"\n[RF] Best PR-AUC: {study_rf.best_value:.4f}")
# print(f"[RF] Best params: {study_rf.best_params}")

### 3-1. RF 대안 최적화 시도 (OOB 기반 탐색 + ExtraTreesClassifier)

Optuna 튜닝이 59만 행 기준 trial당 25분+ 소요로 현실적으로 불가능하다고 판단하여,
동일한 목적(RF 계열 성능 향상)을 빠르게 달성할 수 있는 두 가지 대안을 시도합니다.

**방법 1: OOB Score 기반 빠른 파라미터 탐색**
- RF의 `oob_score=True` 설정 시 CV 없이 out-of-bag 샘플로 성능 추정 가능
- CV 5-fold 대신 OOB score 기준으로 파라미터를 탐색하면 속도가 약 5배 빨라짐
- 유망한 파라미터 조합을 찾은 뒤 CV로 한 번만 최종 검증

**방법 2: ExtraTreesClassifier**
- RF와 구조가 거의 같지만 분기 기준을 무작위로 선택하여 RF보다 학습이 훨씬 빠름
- 실제로 RF와 성능이 비슷하거나 더 좋은 경우가 많아 빠른 대체 후보로 적합
- LightGBM 튜닝 결과(0.5101)를 넘으면 채택, 못 넘으면 스킵

**판단 기준**
- 두 방법 모두 LightGBM 튜닝 결과(PR-AUC 0.5101)를 넘지 못하면
  RF 기본값(0.4972)을 그대로 유지하고 최종 후보 비교로 넘어감

### 참고: Optuna 대신 OOB 기반 탐색 + ExtraTreesClassifier를 선택한 이유

---

#### 왜 처음부터 이 방법을 쓰지 않았는가

초기 설계 단계에서 Optuna(베이지안 최적화)를 선택한 이유는 다음과 같음:
- 베이지안 최적화는 이전 trial 결과를 반영해 탐색 방향을 지능적으로 결정하므로, Random Search보다 효율적임
- 실무에서 하이퍼파라미터 튜닝의 표준 도구로 널리 쓰이며 포트폴리오 가치가 높음

그러나 **데이터 규모(59만 행) × 5-fold CV × trial 수**의 조합에서 RF의 trial당 학습 시간이 25분+로 과도하게 증가하는 문제를 실제로 직면함. 튜닝 방법 선택 전에 "모델별 trial당 예상 학습 시간"을 먼저 계산했어야 했으나, 이 단계를 건너뛰고 도구의 이론적 우수성만 보고 결정한 것이 원인이었음.

이 경험 자체가 실무에서 중요한 교훈임: **좋은 도구도 데이터 규모와 시간 제약을 먼저 고려한 뒤 선택해야 함.**

---

#### 대안 방법 1: OOB(Out-of-Bag) Score 기반 탐색

**OOB Score란**

RF는 각 트리를 학습할 때 전체 데이터에서 복원추출(bootstrap)로 일부 샘플만 사용함.
이때 선택되지 않은 나머지 샘플(Out-of-Bag 샘플)이 자동으로 생기는데,
이 샘플로 해당 트리의 성능을 추정할 수 있음.

즉, **CV(5-fold) 없이도 모델 성능을 추정할 수 있는 RF 고유의 기능**임.

```
전체 데이터 590,540건
    ↓ 복원추출 (bootstrap)
트리 1 학습에 쓰인 샘플: 약 63%
트리 1의 OOB 샘플: 나머지 약 37%  ← 이걸로 트리 1의 성능 추정
(모든 트리의 OOB 추정값 평균 = OOB Score)
```

**왜 빠른가**
- CV 5-fold는 모델을 5번 학습해야 함
- OOB는 모델을 1번만 학습하면 자동으로 검증값이 나옴
- 속도가 약 5배 빨라짐

**주의할 점**
- OOB Score는 기본적으로 accuracy 기반이라 PR-AUC와 척도가 다름
- 따라서 OOB Score는 "좋은 파라미터 후보를 빠르게 걸러내는 용도"로만 쓰고,
  상위 후보는 CV PR-AUC로 최종 검증하는 2단계 방식을 사용함

```
[1단계] OOB Score로 8개 후보 빠르게 평가 (각 2~3분)
    ↓
[2단계] 상위 3개 후보만 CV PR-AUC로 정밀 검증 (각 10~15분)
→ 전체 탐색 시간: Optuna 25분/trial × 15 trial = 375분 → 약 60~70분으로 단축
```

---

#### 대안 방법 2: ExtraTreesClassifier

**RF와의 차이**

RF와 ExtraTrees는 둘 다 여러 개의 결정 트리를 만들어 앙상블하는 방식임.
차이는 **각 트리에서 분기 기준을 선택하는 방식**에 있음.

| 구분 | RF | ExtraTrees |
|---|---|---|
| 분기 기준 선택 | 여러 후보 중 최적값 탐색 | 완전 무작위로 선택 |
| 학습 속도 | 느림 | 빠름 (최적화 탐색 생략) |
| 분산(variance) | 낮음 | 더 낮음 (무작위성 증가) |
| 편향(bias) | 낮음 | 약간 높을 수 있음 |

**왜 빠른가**
RF는 각 분기마다 "여러 후보 feature와 threshold 중 가장 좋은 것"을 찾는 탐색 과정이 있음.
ExtraTrees는 이 탐색 없이 threshold를 무작위로 정해버려서 학습 속도가 RF 대비 2~3배 빠름.

**성능은 어떤가**
무작위성이 높아져서 개별 트리의 정확도는 RF보다 낮을 수 있지만,
앙상블 효과로 전체 성능은 RF와 비슷하거나 더 좋은 경우가 많음.
특히 데이터가 많고 노이즈가 있는 경우(사기탐지처럼)에 ExtraTrees가 유리한 경향이 있음.

**본 프로젝트에서의 활용**
ExtraTrees를 RF의 빠른 대체 후보로 사용하여,
LightGBM 튜닝 결과(PR-AUC 0.5101)와 비교함.
LightGBM을 넘으면 채택, 못 넘으면 RF 기본값(0.4972)을 대표값으로 유지함.

---

#### 이 경험이 주는 실무 교훈

튜닝 방법 선택 전 반드시 확인해야 할 것:
1. 모델별 단일 학습 시간 (데이터 규모 × 모델 복잡도)
2. trial당 예상 시간 = 단일 학습 시간 × CV fold 수
3. 총 튜닝 시간 = trial당 시간 × n_trials

대용량 데이터에서는 Optuna보다 OOB 기반 탐색, 샘플링 후 튜닝,
또는 ExtraTrees처럼 구조적으로 빠른 모델로 대체하는 방식이 더 실용적일 수 있음.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import numpy as np

# OOB score는 PR-AUC와 척도가 다르므로(accuracy 기반),
# OOB score로 파라미터 후보를 빠르게 걸러내고
# 상위 후보만 CV PR-AUC로 최종 검증하는 2단계 방식 사용

param_candidates = [
    {'max_depth': 10, 'min_samples_leaf': 1,  'max_features': 0.3},
    {'max_depth': 15, 'min_samples_leaf': 1,  'max_features': 0.5},
    {'max_depth': 20, 'min_samples_leaf': 2,  'max_features': 0.3},
    {'max_depth': 10, 'min_samples_leaf': 2,  'max_features': 0.5},
    {'max_depth': 15, 'min_samples_leaf': 3,  'max_features': 0.7},
    {'max_depth': 20, 'min_samples_leaf': 1,  'max_features': 0.7},
    {'max_depth': None,'min_samples_leaf': 1, 'max_features': 0.3},
    {'max_depth': None,'min_samples_leaf': 2, 'max_features': 0.5},
]

print("=== OOB Score 기반 빠른 파라미터 탐색 ===")
print("(OOB score는 accuracy 기반이므로 후보 걸러내기 용도로만 사용)\n")

oob_results = []
for i, params in enumerate(param_candidates):
    model = RandomForestClassifier(
        n_estimators=100,
        oob_score=True,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
        **params
    )
    # OOB는 전체 X에 fit (CV 없이 빠르게)
    X_num = X.copy()
    for col in categorical_cols:
        X_num[col] = X_num[col].astype('category').cat.codes

    model.fit(X_num, y)
    oob = model.oob_score_
    oob_results.append({'params': params, 'oob_score': oob})
    print(f"[{i+1}/{len(param_candidates)}] OOB={oob:.4f} | {params}")

# 상위 3개 후보 선정
oob_results_sorted = sorted(oob_results, key=lambda x: x['oob_score'], reverse=True)
top3_params = [r['params'] for r in oob_results_sorted[:3]]
print(f"\n상위 3개 후보 (CV PR-AUC 검증 대상):")
for i, p in enumerate(top3_params):
    print(f"  {i+1}위: {p} (OOB={oob_results_sorted[i]['oob_score']:.4f})")

=== OOB Score 기반 빠른 파라미터 탐색 ===
(OOB score는 accuracy 기반이므로 후보 걸러내기 용도로만 사용)

[1/8] OOB=0.9028 | {'max_depth': 10, 'min_samples_leaf': 1, 'max_features': 0.3}
[2/8] OOB=0.9552 | {'max_depth': 15, 'min_samples_leaf': 1, 'max_features': 0.5}
[3/8] OOB=0.9722 | {'max_depth': 20, 'min_samples_leaf': 2, 'max_features': 0.3}
[4/8] OOB=0.9061 | {'max_depth': 10, 'min_samples_leaf': 2, 'max_features': 0.5}
[5/8] OOB=0.9549 | {'max_depth': 15, 'min_samples_leaf': 3, 'max_features': 0.7}
[6/8] OOB=0.9705 | {'max_depth': 20, 'min_samples_leaf': 1, 'max_features': 0.7}
[7/8] OOB=0.9803 | {'max_depth': None, 'min_samples_leaf': 1, 'max_features': 0.3}
[8/8] OOB=0.9812 | {'max_depth': None, 'min_samples_leaf': 2, 'max_features': 0.5}

상위 3개 후보 (CV PR-AUC 검증 대상):
  1위: {'max_depth': None, 'min_samples_leaf': 2, 'max_features': 0.5} (OOB=0.9812)
  2위: {'max_depth': None, 'min_samples_leaf': 1, 'max_features': 0.3} (OOB=0.9803)
  3위: {'max_depth': 20, 'min_samples_leaf': 2, 'max_features': 0.3} (OOB=0.9

In [14]:
# print("=== 상위 3개 후보 CV PR-AUC 최종 검증 ===\n")

# best_rf_pr_auc = 0.0
# best_rf_params = None

# for i, params in enumerate(top3_params):
#     full_params = {
#         'n_estimators': 300,
#         'class_weight': 'balanced',
#         'n_jobs': -1,
#         'random_state': 42,
#         **params
#     }
#     model = RandomForestClassifier(**full_params)
#     pr_auc = cv_pr_auc(model, use_category=False)
#     print(f"[후보 {i+1}] PR-AUC: {pr_auc:.4f} | params: {params}")

#     if pr_auc > best_rf_pr_auc:
#         best_rf_pr_auc = pr_auc
#         best_rf_params = full_params

# print(f"\n[RF OOB 탐색] Best PR-AUC: {best_rf_pr_auc:.4f}")
# print(f"[RF OOB 탐색] Best params: {best_rf_params}")
# print(f"\nRF 기본값(0.4972) 대비: {best_rf_pr_auc - 0.4972:+.4f}")
# print(f"LightGBM 튜닝(0.5101) 대비: {best_rf_pr_auc - 0.5101:+.4f}")

### 3-2. RF 대안 최적화 진행 방향 수정

**OOB 탐색 결과 분석**
- 상위 3개 후보가 전부 max_depth=None 또는 max_depth=20으로, "깊게 자랄수록 OOB score가 높다"는 경향이 명확하게 나타남
- 다만 OOB score는 accuracy 기반(최고 0.9812)이므로, 사기 비율 3.5%인 불균형 데이터에서 "정상 거래를 전부 맞혀도 96.5%"가 나오는 특성상 OOB score가 높다고 해서 사기 탐지 성능이 좋다는 보장이 없음
- 따라서 OOB score만으로 결론을 내리기 어렵고, PR-AUC 기준 검증이 필요함

**당초 계획(셀 17): 상위 3개 후보 × CV PR-AUC 검증**
- 후보 1개당 n_estimators=300 × 5-fold 풀 CV → 20~30분
- 3개 후보 합산 → 60~90분 소요 예상
- 시간 대비 효율이 낮다고 판단하여 방향 수정

**수정된 진행 방향**
OOB 1위 파라미터(max_depth=None, min_samples_leaf=2, max_features=0.5)를
ExtraTreesClassifier에 적용하여 CV PR-AUC를 1회만 측정하는 방식으로 변경.

이유:
1. ExtraTrees는 RF보다 2~3배 빠르므로 CV 1회가 10~15분 내로 가능함
2. RF와 ExtraTrees는 구조가 거의 같아서 RF 최적 파라미터가 ExtraTrees에도 유사하게 작동함
3. 결과에 따라 명확하게 결론을 낼 수 있음
   - LightGBM 튜닝(0.5101) 초과 시: ExtraTrees 채택
   - LightGBM 튜닝 미만, RF 기본값(0.4972) 초과 시: ExtraTrees를 RF 대표값으로 채택
   - RF 기본값(0.4972) 미만 시: RF 기본값 유지

In [15]:
from sklearn.ensemble import ExtraTreesClassifier

print("=== ExtraTreesClassifier 시도 (OOB 1위 파라미터 적용) ===\n")
print("적용 파라미터: max_depth=None, min_samples_leaf=2, max_features=0.5")
print("(OOB Score 기반 탐색 1위 결과 적용)\n")

model_et = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,        # OOB 1위
    min_samples_leaf=2,    # OOB 1위
    max_features=0.5,      # OOB 1위
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

pr_auc_et = cv_pr_auc(model_et, use_category=False)
print(f"[ExtraTrees] PR-AUC: {pr_auc_et:.4f}")
print(f"RF 기본값(0.4972) 대비: {pr_auc_et - 0.4972:+.4f}")
print(f"LightGBM 튜닝(0.5101) 대비: {pr_auc_et - 0.5101:+.4f}")

=== ExtraTreesClassifier 시도 (OOB 1위 파라미터 적용) ===

적용 파라미터: max_depth=None, min_samples_leaf=2, max_features=0.5
(OOB Score 기반 탐색 1위 결과 적용)

[ExtraTrees] PR-AUC: 0.4912
RF 기본값(0.4972) 대비: -0.0060
LightGBM 튜닝(0.5101) 대비: -0.0189


### 3-3. RF 대안 최적화 결과 정리

**ExtraTreesClassifier 결과**
- 적용 파라미터: max_depth=None, min_samples_leaf=2, max_features=0.5 (OOB 1위)
- PR-AUC: 0.4912
- RF 기본값(0.4972) 대비: -0.0060 (하회)
- LightGBM 튜닝(0.5101) 대비: -0.0189 (하회)

**판단**
ExtraTrees가 RF 기본값보다도 낮게 나와 탈락. RF 기본값(0.4972) 유지.

**RF 대안 최적화 전체 시도 결과 요약**

| 시도 | 방법 | PR-AUC | 결과 |
|---|---|---|---|
| RF 기본값 | n_estimators=100, 기본 파라미터 | 0.4972 | 기준값 |
| Optuna 튜닝 | n_estimators=50~200, 5 fold CV | 진행 불가 | trial당 25분+ 소요 |
| OOB 기반 탐색 | 8개 후보 OOB score 비교 | - | 후보 걸러내기 완료 |
| ExtraTrees(OOB 1위) | max_depth=None, min_samples_leaf=2, max_features=0.5 | 0.4912 | RF 기본값 하회 → 탈락 |

**최종 결론**
RF 계열 모든 대안 시도 결과, RF 기본값(0.4972)이 가장 높은 성능을 보임.
RF 기본값을 RF 대표값으로 확정하며, 최종 후보는 다음 두 모델로 좁혀짐.

- **최종 후보 1**: LightGBM (튜닝 후 PR-AUC 0.5101)
- **최종 후보 2**: RF (기본값 PR-AUC 0.4972)

다음 단계: CalibratedClassifierCV 확률 보정 후 두 모델의 최종 성능을 비교하여 최종 모델을 선정함.

### 참고: OOB 기반 탐색과 ExtraTrees는 RF 전용인가?

---

#### OOB Score 기반 탐색 — RF 계열에서만 사용 가능

OOB는 RF가 bootstrap 샘플링(복원추출)을 사용하기 때문에 자동으로 생기는 부산물임.
LightGBM, XGBoost, Logistic Regression 등은 bootstrap 방식을 쓰지 않아서 OOB 자체가 존재하지 않음.

정확히는 sklearn의 bootstrap 기반 앙상블 계열
(RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier 등)에서만 사용 가능함.

---

#### ExtraTreesClassifier — RF 대체용으로만 의미 있음

ExtraTrees는 RF의 변형 모델이라, LightGBM이나 XGBoost를 ExtraTrees로 대체하는 건
"튜닝"이 아니라 "완전히 다른 모델로 교체"하는 것임. RF 계열 내에서의 대안으로만 적합함.

---

#### LightGBM / XGBoost에서 Optuna 없이 빠르게 튜닝하는 방법

LightGBM과 XGBoost는 OOB나 ExtraTrees 같은 RF 전용 방법을 쓸 수 없지만,
오히려 더 실용적인 대안들이 있음.

**1. Early Stopping (이미 적용)**
- n_estimators를 크게 잡고, validation 성능이 수렴하면 자동 중단
- 본 프로젝트에서 n_estimators=5000 + early_stopping_rounds=50으로 적용함
- 불필요한 트리 추가 없이 best_iteration을 자동으로 찾아줌

**2. Learning Rate + n_estimators 연동**
- learning_rate를 작게 고정(예: 0.05)하고 early stopping으로 최적 n_estimators를 탐색
- "learning_rate가 작을수록 트리가 많이 필요하지만 성능이 안정적"이라는 특성을 활용
- 파라미터 탐색 공간을 크게 줄일 수 있어 속도 향상

**3. HalvingGridSearchCV (sklearn)**
- 연속 반감법(Successive Halving) 기반의 탐색 방식
- 초기엔 적은 데이터로 많은 후보를 빠르게 평가하고, 라운드마다 데이터를 늘려가며 후보를 절반씩 탈락시킴
- Random Search보다 빠르고, Optuna보다 구현이 단순하여 빠른 탐색에 적합
- LightGBM, XGBoost, 어떤 sklearn 호환 모델에도 사용 가능

```
[1라운드] 전체 데이터의 10%로 100개 후보 평가 → 하위 50% 탈락
[2라운드] 전체 데이터의 20%로 50개 후보 평가 → 하위 50% 탈락
[3라운드] 전체 데이터의 40%로 25개 후보 평가 → 하위 50% 탈락
...
[최종] 전체 데이터로 최종 후보 평가
```

**4. Optuna + 샘플링 조합**
- Optuna의 베이지안 최적화는 유지하되, 튜닝 중에는 전체 데이터의 30~40%만 샘플링하여 trial 속도를 높이는 방식
- best params를 찾은 뒤 전체 데이터로 최종 재학습

---

#### 정리: 모델별 실용적 튜닝 전략

| 모델 | 추천 튜닝 방법 | 이유 |
|---|---|---|
| RF | OOB 기반 탐색 + ExtraTrees 대체 검토 | Optuna trial당 시간 과다, OOB로 CV 대체 가능 |
| LightGBM | Optuna + Early Stopping | 학습 빠르고 탐색축 많아 베이지안 최적화 효과 큼 |
| XGBoost | Optuna + Early Stopping 또는 HalvingGridSearchCV | LightGBM과 유사하나 상대적으로 느림 |

핵심 원칙: **튜닝 방법 선택 전, 모델별 단일 학습 시간과 총 튜닝 예상 시간을 먼저 계산할 것**

### 4. LightGBM 튜닝 (20 trials, 경량화 버전)

**초기 설정 대비 변경 사항**
- n_trials: 50 → 20 (RF가 이미 1위인 상황에서 "RF를 넘길 수 있는지 빠르게 확인"하는 용도로 축소)
- n_estimators: 100~1000 탐색 → 5000 고정 + LightGBM early stopping으로 best_iteration 자동 탐색
- search space: 8개 파라미터 → 핵심 6개로 축소 (max_depth, reg_alpha 제거)
- 이유: 59만 행 + 5-fold CV에서 넓은 search space는 trial당 20~30분으로 전체 튜닝이 수 시간에 달함.
  실무적으로 "많이 찾는 방식"보다 "빨리 후보를 걸러내는 방식"이 더 효율적임

**탐색 파라미터**
- learning_rate: 학습률 (핵심)
- num_leaves: 리프 노드 최대 수 (LightGBM 핵심 파라미터)
- min_child_samples: 리프 노드 최소 샘플 수
- subsample: 각 트리에 사용할 데이터 비율
- colsample_bytree: 각 트리에서 사용할 feature 비율
- reg_lambda: L2 정규화

**고정 파라미터**
- n_estimators=5000 (early stopping이 best_iteration을 자동으로 찾으므로 크게 설정)
- scale_pos_weight=imbalance_ratio (Day 4에서 확정)

In [7]:
import lightgbm as lgb

def cv_pr_auc_lgbm_es(params):
    """
    LightGBM early stopping 적용 버전 CV.
    각 fold에서 validation set을 early stopping 기준으로 사용.
    """
    sgkf = StratifiedGroupKFold(n_splits=5)
    pr_aucs = []

    for train_idx, val_idx in sgkf.split(X, y, groups=groups):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val   = y.iloc[val_idx]

        scaler = StandardScaler()
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_val[scale_cols]   = scaler.transform(X_val[scale_cols])

        for col in categorical_cols:
            X_train[col] = X_train[col].astype('category')
            X_val[col]   = X_val[col].astype('category')

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='average_precision',
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=-1)  # 로그 출력 억제
            ]
        )

        y_pred = model.predict_proba(X_val)[:, 1]
        pr_aucs.append(average_precision_score(y_val, y_pred))

    return np.mean(pr_aucs)


def objective_lgbm(trial):
    params = {
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators':     5000,
        'scale_pos_weight': imbalance_ratio,
        'random_state':     42,
        'verbosity':        -1,
    }
    return cv_pr_auc_lgbm_es(params)


study_lgbm = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_lgbm.optimize(objective_lgbm, n_trials=20, show_progress_bar=True)

print(f"\n[LightGBM] Best PR-AUC: {study_lgbm.best_value:.4f}")
print(f"[LightGBM] Best params: {study_lgbm.best_params}")

Best trial: 17. Best value: 0.510089: 100%|██████████| 20/20 [1:02:47<00:00, 188.39s/it]


[LightGBM] Best PR-AUC: 0.5101
[LightGBM] Best params: {'learning_rate': 0.1563037874293778, 'num_leaves': 165, 'min_child_samples': 32, 'subsample': 0.9248534606596772, 'colsample_bytree': 0.6651160762107319, 'reg_lambda': 0.00025077983643406634}


### 5. XGBoost 튜닝 (20 trials, 경량화 버전)

LightGBM과 동일한 경량화 전략을 적용합니다.

**변경 사항**
- n_trials: 50 → 20
- n_estimators: 5000 고정 + early stopping
- search space: 핵심 6개 파라미터만 탐색
- scale_pos_weight: 고정값(27.6) 대신 탐색 범위 설정
  (Day 5 Calibration 분석에서 과신 문제 확인 → 최적값 탐색 필요)

### 6. 튜닝 결과 정리 및 저장

**튜닝 진행 결과 요약**
- LightGBM: 20 trials 완료 → PR-AUC 0.4860 → 0.5101 (+0.0241 향상)
- RF: 대용량 데이터(59만 행)에서 trial당 학습 시간이 과도하게 증가하여 튜닝 보류.
  기본값(PR-AUC 0.4972) 유지
- XGBoost: RF와 동일한 이유로 튜닝 보류. 기본값(PR-AUC 0.4471) 유지

**최종 후보 확정**
- LightGBM(튜닝) vs RF(기본값) 2개 모델로 좁혀서 CalibratedClassifierCV 보정 후 최종 선정
- RF 튜닝 보류 이유: 대용량 데이터에서 학습 속도가 느린 RF의 실용적 한계로,
  튜닝 효율이 높고 학습 속도가 빠른 LightGBM에 튜닝 리소스를 집중하는 것이 합리적 판단임

**튜닝 전후 PR-AUC 비교**

| 모델 | 기본값 | 튜닝 후 | 향상 |
|---|---|---|---|
| LightGBM | 0.4860 | **0.5101** | +0.0241 |
| RF | 0.4972 | 튜닝 보류 | - |
| XGBoost | 0.4471 | 튜닝 보류 | - |

In [11]:
import pickle, os

tuning_results = {
    'study_lgbm': study_lgbm,
    'baseline': {
        'Random Forest':  0.4972,
        'LightGBM':       0.4860,
        'XGBoost':        0.4471,
    },
    'tuned': {
        'LightGBM': study_lgbm.best_value,
    },
    'best_params': {
        'LightGBM': study_lgbm.best_params,
    },
    'memo': {
        'RF':      'n_estimators=300 고정, 59만 행 기준 trial당 25분+ 소요로 튜닝 보류. 기본값 유지.',
        'XGBoost': 'RF와 동일한 이유로 튜닝 보류. 기본값 유지.',
    }
}

os.makedirs("../data/processed", exist_ok=True)
with open("../data/processed/day6_tuning_results.pkl", 'wb') as f:
    pickle.dump(tuning_results, f)

print("저장 완료")
print(f"\n[LightGBM] Best PR-AUC: {study_lgbm.best_value:.4f}")
print(f"[LightGBM] Best params:")
for k, v in study_lgbm.best_params.items():
    print(f"  {k}: {v}")

저장 완료

[LightGBM] Best PR-AUC: 0.5101
[LightGBM] Best params:
  learning_rate: 0.1563037874293778
  num_leaves: 165
  min_child_samples: 32
  subsample: 0.9248534606596772
  colsample_bytree: 0.6651160762107319
  reg_lambda: 0.00025077983643406634
